# Lesson 7.5 - Capstone: End-to-End Agentic AI System (Template)

## Learning Objectives
- Scope an end-to-end agentic capstone with realistic constraints.
- Use reusable templates for architecture, evaluation, and monitoring.
- Implement a toy RAG + agent loop skeleton that is easy to extend.


## Capstone Brief & Checklist Template
```text
Problem statement:
Primary user persona:
Business objective and KPI:
Data sources / corpus:
Agent capabilities and tools:
Safety constraints and escalation policy:
Deployment target:
Evaluation plan:
```

### Readiness Checklist
- [ ] Clear scope boundaries (what is in/out).
- [ ] Baseline before agentic complexity.
- [ ] Observable logs and quality metrics defined.
- [ ] Safety and fallback behavior specified.
- [ ] Demo script prepared for stakeholders.


In [1]:
from dataclasses import dataclass
from typing import List, Dict


@dataclass
class CapstonePlan:
    problem: str
    kpi: str
    tools: List[str]
    risks: List[str]


def score_plan(plan: CapstonePlan) -> Dict[str, int]:
    return {
        'clarity': 2 if len(plan.problem) > 30 else 1,
        'measurability': 2 if '%' in plan.kpi or 'rate' in plan.kpi.lower() else 1,
        'operational_risk': max(1, 3 - min(len(plan.risks), 2)),
        'tooling_coverage': 2 if len(plan.tools) >= 2 else 1,
    }


plan = CapstonePlan(
    problem='Answer enterprise policy questions from internal docs with citations.',
    kpi='Answer acceptance rate >= 80%',
    tools=['retriever', 'calculator'],
    risks=['hallucination', 'stale index'],
)
print(score_plan(plan))


{'clarity': 2, 'measurability': 2, 'operational_risk': 1, 'tooling_coverage': 2}


## Tiny RAG Pipeline Skeleton


In [2]:
CORPUS = [
    'Refund policy: refunds allowed within 30 days with receipt.',
    'Security policy: all employees must rotate passwords every 90 days.',
    'Remote-work policy: managers approve hybrid schedules quarterly.',
]


def chunk_documents(docs, chunk_size=8):
    chunks = []
    for d in docs:
        tokens = d.split()
        for i in range(0, len(tokens), chunk_size):
            chunks.append(' '.join(tokens[i:i+chunk_size]))
    return chunks


def retrieve(query: str, chunks: list[str], top_k: int = 2):
    q_terms = set(query.lower().split())
    scored = []
    for ch in chunks:
        overlap = len(q_terms.intersection(set(ch.lower().split())))
        scored.append((overlap, ch))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for s, c in scored[:top_k] if s > 0]


def generate_answer(query: str, contexts: list[str]) -> str:
    if not contexts:
        return 'I do not have grounded evidence in the current index.'
    return f"Query: {query}\nGrounded context: {' | '.join(contexts)}"

chunks = chunk_documents(CORPUS)
ctx = retrieve('what is the refund policy', chunks)
print(generate_answer('what is the refund policy', ctx))


Query: what is the refund policy
Grounded context: Refund policy: refunds allowed within 30 days with


## Agent Loop Skeleton
Use explicit planner -> executor -> verifier phases to keep behavior inspectable.


In [3]:
def planner(user_goal: str) -> list[str]:
    return ['retrieve_context', 'draft_answer', 'self_check']


def executor(step: str, state: dict) -> dict:
    if step == 'retrieve_context':
        state['contexts'] = retrieve(state['query'], chunk_documents(CORPUS))
    elif step == 'draft_answer':
        state['draft'] = generate_answer(state['query'], state.get('contexts', []))
    elif step == 'self_check':
        state['safe'] = 'Grounded context:' in state.get('draft', '')
    return state


def run_agent(query: str) -> dict:
    state = {'query': query}
    for step in planner(query):
        state = executor(step, state)
    return state

print(run_agent('password rotation policy'))


{'query': 'password rotation policy', 'contexts': [], 'draft': 'I do not have grounded evidence in the current index.', 'safe': False}


## Monitoring Hooks Template
Track these events from day one:
- request metadata (timestamp, user type, route)
- retrieval metrics (hit rate, top-k overlap proxy)
- generation quality signal (thumbs up/down, acceptance)
- safety flags (policy violations, escalation events)


In [4]:
event_log = [
    {'type': 'request', 'latency_ms': 420, 'tokens': 650, 'accepted': True},
    {'type': 'request', 'latency_ms': 810, 'tokens': 1400, 'accepted': False},
    {'type': 'request', 'latency_ms': 510, 'tokens': 720, 'accepted': True},
]


def summarize_events(events):
    n = len(events)
    avg_latency = sum(e['latency_ms'] for e in events) / n
    accept_rate = sum(1 for e in events if e['accepted']) / n
    avg_tokens = sum(e['tokens'] for e in events) / n
    return {'avg_latency_ms': round(avg_latency, 2), 'accept_rate': round(accept_rate, 3), 'avg_tokens': round(avg_tokens, 1)}

print(summarize_events(event_log))


{'avg_latency_ms': 580.0, 'accept_rate': 0.667, 'avg_tokens': 923.3}


## Business Case Studies & Exceptions
### Case 1: Enterprise Policy Assistant
A team launched a RAG assistant for HR/legal policies. Success depended less on model size and more on index quality, source freshness, and explicit citations.

### Case 2: Agentic Workflow for Support Ops
A coordinator agent delegated triage, retrieval, and response drafting. Performance improved only after adding strict tool permissions and escalation to humans for high-risk tickets.

### Exception
If workflow is mostly deterministic and low-variance, a rules-based pipeline with selective LLM calls is often safer and cheaper than a full autonomous agent.


## Interview Questions & Answers
1. **Q:** How would you scope an agentic capstone to be realistic?  
   **A:** Limit to one core user workflow with measurable KPI and explicit non-goals.
2. **Q:** Why start with a baseline before agents?  
   **A:** To prove incremental value of orchestration complexity.
3. **Q:** RAG vs fine-tuning for capstone?  
   **A:** RAG is often faster and easier for knowledge grounding with changing documents.
4. **Q:** What telemetry is essential in agentic apps?  
   **A:** Latency, token/cost usage, retrieval quality proxies, acceptance/safety events.
5. **Q:** How do you handle tool misuse risk?  
   **A:** Tool permissions, policy checks, and human escalation gates.
6. **Q:** What makes a capstone interview-ready?  
   **A:** Clear problem framing, architecture decisions, metrics, trade-offs, and lessons learned.
7. **Q:** Why include a self-check step in agent loops?  
   **A:** To validate grounding/format/safety before final output.
8. **Q:** One common failure mode in RAG?  
   **A:** Retriever misses relevant context due to poor chunking/indexing.
9. **Q:** How do you demonstrate business value?  
   **A:** Report KPI deltas (time saved, acceptance rate, error reduction).
10. **Q:** When is full agent autonomy a bad idea?  
   **A:** High-risk domains without robust guardrails and human oversight.


## End-to-End Delivery Checklist
- [ ] Problem/KPI baseline recorded
- [ ] Retrieval and grounding quality tested
- [ ] Agent actions bounded by policy constraints
- [ ] Monitoring traces captured for each major step
- [ ] Demo and rollback path documented


## Capstone Rubric (Agentic Systems)
Use weighted scoring:
- Scope & KPI clarity (15%)
- Architecture & implementation (25%)
- Evaluation & reliability (20%)
- Ops/governance readiness (20%)
- Communication & reflection (20%)
